### Загрузка и сплит датасета

In [1]:
from typing import Literal
from datasets import Dataset, DatasetDict, load_dataset
import pandas as pd
import scipy.sparse as sparse
import implicit

class YambdaDataset:
    INTERACTIONS = frozenset([
        "likes", "listens", "multi_event", "dislikes", "unlikes", "undislikes"
    ])

    def __init__(
        self,
        dataset_type: Literal["flat", "sequential"] = "flat",
        dataset_size: Literal["50m", "500m", "5b"] = "50m"
    ):
        assert dataset_type in {"flat", "sequential"}
        assert dataset_size in {"50m", "500m", "5b"}
        self.dataset_type = dataset_type
        self.dataset_size = dataset_size

    def interaction(self, event_type: Literal[
        "likes", "listens", "multi_event", "dislikes", "unlikes", "undislikes"
    ]) -> Dataset:
        assert event_type in YambdaDataset.INTERACTIONS
        return self._download(f"{self.dataset_type}/{self.dataset_size}", event_type)

    def audio_embeddings(self) -> Dataset:
        return self._download("", "embeddings")

    def album_item_mapping(self) -> Dataset:
        return self._download("", "album_item_mapping")

    def artist_item_mapping(self) -> Dataset:
        return self._download("", "artist_item_mapping")

    @staticmethod
    def _download(data_dir: str, file: str) -> Dataset:
        data = load_dataset("yandex/yambda", data_dir=data_dir, data_files=f"{file}.parquet")
        # Returns DatasetDict; extracting the only split
        assert isinstance(data, DatasetDict)
        return data["train"]

dataset = YambdaDataset("flat", "50m")
multi_events = dataset.interaction("multi_event")
multi_events_df = multi_events.to_pandas()  # returns a HuggingFace Dataset


/opt/anaconda3/envs/for_hw/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def split_data_by_timestamp(df: pd.DataFrame, train_threshold: float = 0.8):
    """
    Splits a DataFrame into training and testing sets based on timestamp.

    The DataFrame is sorted by 'timestamp', and then split based on the
    train_threshold. The training set will contain older data, and the
    testing set will contain newer data.

    Args:
        df (pd.DataFrame): The input DataFrame containing a 'timestamp' column.
        train_threshold (float): The proportion of data to use for the training set.

    Returns:
        tuple: A tuple containing (train_df, test_df).
    """
    # Sort the DataFrame by timestamp to ensure a chronological split
    df_sorted = df.sort_values(by='timestamp').reset_index(drop=True)

    # Calculate the number of rows for the training set
    train_size = int(len(df_sorted) * train_threshold)

    # Split the DataFrame into training and testing sets
    train_df = df_sorted.iloc[:train_size]
    test_df = df_sorted.iloc[train_size:]

    print(f"Training set size: {len(train_df)} rows")
    print(f"Testing set size: {len(test_df)} rows")

    return train_df, test_df

# Define the split ratio
TRAIN_THRESHOLD = 0.8

# Split the DataFrame using the new function
train_df, test_df = split_data_by_timestamp(multi_events_df, TRAIN_THRESHOLD)

Training set size: 38232359 rows
Testing set size: 9558090 rows


In [3]:
train_df.to_parquet("train.parquet")
test_df.to_parquet("test.parquet")

train_df

,uid,timestamp,item_id,is_organic,played_ratio_pct,track_length_seconds,event_type
0,468300,0,7400764,0,100.0,225.0,listen
1,347600,5,3415205,0,100.0,250.0,listen
2,942900,10,6728180,0,1.0,270.0,listen
3,12700,15,8932363,0,100.0,245.0,listen
4,243500,15,5283544,1,100.0,195.0,listen
...,...,...,...,...,...,...,...
38232354,694300,22128035,3142020,1,100.0,110.0,listen
38232355,694300,22128035,4104848,1,100.0,135.0,listen
38232356,694300,22128035,8853148,1,100.0,220.0,listen
38232357,694300,22128035,6463706,1,100.0,150.0,listen


### `ALSCombiner` Class for ALS Embeddings

This class provides a structured way to train an Alternating Least Squares (ALS) model using the `implicit` library, extract user and item embeddings, and save them for later use. It handles the conversion of interaction data into a sparse matrix format required by ALS.

In [29]:
from typing import Literal
from datasets import Dataset, DatasetDict, load_dataset
import pandas as pd
import scipy.sparse as sparse
import implicit
import os
import joblib

class ALSCombiner:
    """
    A class to generate user and item embeddings using the Alternating Least Squares (ALS)
    algorithm from the `implicit` library and save them to files.
    """
    def __init__(self, decomposition_name: str, factors: int = 64, regularization: float = 0.01,
                 iterations: int = 20, random_state: int = 42):
        self.decomposition_name = decomposition_name
        self.factors = factors
        self.regularization = regularization
        self.iterations = iterations
        self.random_state = random_state
        self.model = implicit.als.AlternatingLeastSquares(
            factors=self.factors,
            regularization=self.regularization,
            iterations=self.iterations,
            random_state=self.random_state
        )
        self.user_to_idx = None
        self.item_to_idx = None
        self.user_original_ids = None
        self.item_original_ids = None
        self.user_embeddings_df = None
        self.item_embeddings_df = None
        self.user_item_matrix = None # Store the user-item matrix for recommendations

        # Set environment variable to avoid OpenBLAS performance issues
        os.environ['OPENBLAS_NUM_THREADS'] = '1'

    def _prepare_data(self, df: pd.DataFrame, user_col: str = 'uid', item_col: str = 'item_id',
                     event_type_col: str = 'event_type',
                     event_weights: dict = None, listen_weight: float = 0.01):
        """
        Prepares the data for ALS training by creating a sparse user-item matrix
        and ID mappings, applying specified weights based on event_type.
        """
        # Create mappings for user and item IDs to contiguous integers
        user_ids_category = df[user_col].astype("category")
        item_ids_category = df[item_col].astype("category")

        self.user_to_idx = {user_id: idx for idx, user_id in enumerate(user_ids_category.cat.categories)}
        self.item_to_idx = {item_id: idx for idx, item_id in enumerate(item_ids_category.cat.categories)}

        self.user_original_ids = user_ids_category.cat.categories.to_list()
        self.item_original_ids = item_ids_category.cat.categories.to_list()

        if event_weights is None:
            # Default weights for other interaction types, matching YambdaDataset.INTERACTIONS
            event_weights = {"likes": 1.0, "dislikes": -1.0, "unlikes": -1.0, "undislikes": -1.0}

        # Initialize data array with default value (e.g., 0.0 for interactions not explicitly handled)
        data = pd.Series([0.0] * len(df), index=df.index)

        # Apply listen_weight for 'listen' events
        listen_mask = df[event_type_col] == 'listen'
        data.loc[listen_mask] = df.loc[listen_mask, 'played_ratio_pct'] * listen_weight

        # Apply event_weights for other specific event types
        for event, weight in event_weights.items():
            event_mask = df[event_type_col] == event
            data.loc[event_mask] = weight

        rows = user_ids_category.cat.codes  # Mapped user indices
        cols = item_ids_category.cat.codes  # Mapped item indices

        # Create the user-item sparse matrix (CSR format for efficient row slicing)
        user_item_matrix = sparse.csr_matrix(
            (data.values, (rows, cols)),
            shape=(len(self.user_original_ids), len(self.item_original_ids))
        )
        return user_item_matrix

    def fit(self, df: pd.DataFrame, user_col: str = 'uid', item_col: str = 'item_id',
            event_type_col: str = 'event_type',
            event_weights: dict = None, listen_weight: float = 0.01):
        """
        Trains the ALS model on the provided DataFrame.
        """
        print("Preparing data for ALS model training with custom event weights...")
        self.user_item_matrix = self._prepare_data(df, user_col, item_col,
                                             event_type_col=event_type_col,
                                             event_weights=event_weights,
                                             listen_weight=listen_weight)

        # Train on user-item matrix directly (users x items)
        print("Training ALS model...")
        self.model.fit(self.user_item_matrix)
        print("ALS model trained successfully!")

    def get_embeddings(self):
        """
        Extracts user and item embeddings from the trained ALS model.
        """
        if self.model is None:
            raise ValueError("Model has not been trained. Call .fit() first.")

        # User embeddings are from model.user_factors (as model trained on user-item matrix)
        user_embeddings = self.model.user_factors
        user_embeddings_df = pd.DataFrame(user_embeddings, index=self.user_original_ids)
        user_embeddings_df.index.name = 'uid'
        self.user_embeddings_df = user_embeddings_df.apply(lambda x: x.tolist(), axis=1).to_frame(name=self.decomposition_name)
        self.user_embeddings_df.index.name = user_embeddings_df.index.name

        # Item embeddings are from model.item_factors (as model trained on user-item matrix)
        item_embeddings = self.model.item_factors
        item_embeddings_df = pd.DataFrame(item_embeddings, index=self.item_original_ids)
        item_embeddings_df.index.name = 'item_id'
        self.item_embeddings_df = item_embeddings_df.apply(lambda x: x.tolist(), axis=1).to_frame(name=self.decomposition_name)
        self.item_embeddings_df.index.name = item_embeddings_df.index.name

        return self.user_embeddings_df, self.item_embeddings_df

    def save_embeddings(self, user_embeddings_path: str, item_embeddings_path: str):
        """
        Saves the generated user and item embeddings to specified file paths (parquet format).
        If a file exists, it's loaded and new embeddings are added as a new column.
        """
        if self.user_embeddings_df is None or self.item_embeddings_df is None:
            user_embeddings, item_embeddings = self.get_embeddings()
        else:
            user_embeddings = self.user_embeddings_df
            item_embeddings = self.item_embeddings_df

        # Save User Embeddings
        print(f"Saving user embeddings to {user_embeddings_path}...")
        if os.path.exists(user_embeddings_path):
            existing_user_embeddings = pd.read_parquet(user_embeddings_path)
            merged_user_embeddings = existing_user_embeddings.merge(
                user_embeddings,
                left_index=True,
                right_index=True,
                how='outer'
            )
            merged_user_embeddings.to_parquet(user_embeddings_path)
        else:
            user_embeddings.to_parquet(user_embeddings_path)
        print(f"User embeddings saved successfully for decomposition: {self.decomposition_name}")

        # Save Item Embeddings
        print(f"Saving item embeddings to {item_embeddings_path}...")
        if os.path.exists(item_embeddings_path):
            existing_item_embeddings = pd.read_parquet(item_embeddings_path)
            merged_item_embeddings = existing_item_embeddings.merge(
                item_embeddings,
                left_index=True,
                right_index=True,
                how='outer'
            )
            merged_item_embeddings.to_parquet(item_embeddings_path)
        else:
            item_embeddings.to_parquet(item_embeddings_path)
        print(f"Item embeddings saved successfully for decomposition: {self.decomposition_name}")

    def save_model(self, directory: str = "."):
        """
        Saves the trained ALS model and its associated state to the specified directory using joblib,
        using the decomposition_name as part of the filename.
        """
        if self.model is None:
            raise ValueError("Model has not been trained. Call .fit() first.")

        model_filename = f"{self.decomposition_name}.joblib"
        model_path = os.path.join(directory, model_filename)

        # Save all relevant attributes for full model reconstruction
        model_state = {
            'model': self.model,
            'user_to_idx': self.user_to_idx,
            'item_to_idx': self.item_to_idx,
            'user_original_ids': self.user_original_ids,
            'item_original_ids': self.item_original_ids,
            'user_item_matrix': self.user_item_matrix,
            'decomposition_name': self.decomposition_name,
            'factors': self.factors,
            'regularization': self.regularization,
            'iterations': self.iterations,
            'random_state': self.random_state
        }

        os.makedirs(directory, exist_ok=True)
        print(f"Saving ALS model and state to {model_path}...")
        joblib.dump(model_state, model_path)
        print(f"ALS model and state saved successfully as {model_filename}!")

    def recommend_items(self, user_id: int, N: int = 10) -> pd.DataFrame:
        """
        Recommends N items for a given user_id.
        """
        if self.model is None:
            raise ValueError("Model has not been trained. Call .fit() first.")
        if self.user_to_idx is None or self.item_original_ids is None:
            raise ValueError("Data mappings (user_to_idx, item_original_ids) are not available. Call .fit() first.")

        if user_id not in self.user_to_idx:
            print(f"Warning: User ID {user_id} not found in training data mappings. Cannot provide recommendations.")
            return pd.DataFrame(columns=['item_id', 'score'])

        user_idx = self.user_to_idx[user_id]

        # Explicitly get the user's interaction row from the user-item matrix
        # The 'implicit' library's recommend method expects the specific user's interaction row,
        # not the entire user-item matrix, for the 'user_items' parameter when 'userid' is a single integer.
        user_interaction_row = self.user_item_matrix[user_idx]

        # Use the implicit library's recommend method
        recommendations = self.model.recommend(
            user_idx,
            user_interaction_row, # Pass only the specific user's interaction row
            N=N,
            filter_already_liked_items=True
        )

        # recommendations is a tuple of (item_indices, scores)
        recommended_item_indices = recommendations[0]
        scores = recommendations[1]

        # Convert internal item indices back to original item_ids
        recommended_item_ids = [self.item_original_ids[idx] for idx in recommended_item_indices]

        return pd.DataFrame({'item_id': recommended_item_ids, 'score': scores})

    @classmethod
    def from_pretrained(cls, model_path: str):
        """
        Loads a previously saved ALSCombiner model and its state from the specified path.
        """
        print(f"Loading ALS model and state from {model_path}...")
        model_state = joblib.load(model_path)

        # Reconstruct ALSCombiner instance
        instance = cls(
            decomposition_name=model_state['decomposition_name'],
            factors=model_state['factors'],
            regularization=model_state['regularization'],
            iterations=model_state['iterations'],
            random_state=model_state['random_state']
        )
        instance.model = model_state['model']
        instance.user_to_idx = model_state['user_to_idx']
        instance.item_to_idx = model_state['item_to_idx']
        instance.user_original_ids = model_state['user_original_ids']
        instance.item_original_ids = model_state['item_original_ids']
        instance.user_item_matrix = model_state['user_item_matrix']

        print(f"ALS model and state loaded successfully from {model_path}!")
        return instance


### `TopPop` Class for Popularity-Based Recommendations

This class implements a simple popularity-based recommender. It calculates the popularity of items based on their frequency in the training data and recommends the top `k` most popular items.

In [30]:
class TopPop:
    """
    A simple popularity-based recommender system.
    It recommends the top-k most popular items based on their frequency in the training data.
    """

    def __init__(self):
        self.item_popularity = None

    def fit(self, train_df: pd.DataFrame, item_col: str = 'item_id', event_type_col: str = 'event_type'):
        """
        Calculates the popularity of items based on their occurrences in the training data.
        Popularity is defined as the count of 'listen' events for each item.

        Args:
            train_df (pd.DataFrame): The training DataFrame containing item interactions.
            item_col (str): The name of the column representing item IDs.
            event_type_col (str): The name of the column representing event types.
        """
        print("Fitting TopPop model...")
        # Filter for 'listen' events and count occurrences of each item
        listen_events = train_df[train_df[event_type_col] == 'listen']
        self.item_popularity = listen_events[item_col].value_counts().reset_index()
        self.item_popularity.columns = [item_col, 'popularity_score']
        self.item_popularity = self.item_popularity.sort_values(by='popularity_score', ascending=False)
        print("TopPop model fitted successfully!")

    def recommend_items(self, user_id: int = None, N: int = 10) -> pd.DataFrame:
        """
        Recommends the top N most popular items.
        The `user_id` parameter is included for API compatibility but is not used in this model.

        Args:
            user_id (int): The ID of the user for whom to generate recommendations (ignored in TopPop).
            N (int): The number of top popular items to recommend.

        Returns:
            pd.DataFrame: A DataFrame with 'item_id' and 'popularity_score' for the top N items.
        """
        if self.item_popularity is None:
            raise ValueError("TopPop model has not been fitted. Call .fit() first.")

        return self.item_popularity.head(N).copy()

    def get_top_k_items(self, k: int = 10) -> pd.DataFrame:
        """
        Alias for recommend_items to match the user's request phrasing.
        """
        return self.recommend_items(N=k)



In [31]:
import numpy as np
import pandas as pd
import sys

def add_als_dot_product_features(
    df: pd.DataFrame,
    als_decomposition_names: list[str],
    user_embeddings_path: str,
    item_embeddings_path: str
) -> pd.DataFrame:
    """
    Adds new features to the DataFrame based on the scalar product of ALS user and item embeddings.
    Optimized for memory by using df.apply with direct map lookups instead of intermediate large Series of lists.

    Args:
        df (pd.DataFrame): The input DataFrame (e.g., train_df) containing 'uid' and 'item_id'.
        als_decomposition_names (list[str]): A list of ALS decomposition names (column names
                                            in the embeddings files) to use for feature creation.
        user_embeddings_path (str): Path to the user embeddings parquet file.
        item_embeddings_path (str): Path to the item embeddings parquet file.

    Returns:
        pd.DataFrame: The DataFrame with added scalar product features.
    """
    df_with_features = df
    print(f"Размер исходного df_with_features: {sys.getsizeof(df_with_features) // 1024 // 1024}MB")

    # Вспомогательная функция для df.apply
    def _calculate_dot_product_for_row(row, user_map, item_map, factors):
        uid = row['uid']
        item_id = row['item_id']

        user_emb = user_map.get(uid)
        item_emb = item_map.get(item_id)

        if user_emb is not None and item_emb is not None:
            # Преобразуем списки в массивы numpy только для текущей строки
            return np.dot(np.array(user_emb), np.array(item_emb))
        else:
            return 0.0 # Значение по умолчанию для отсутствующих эмбеддингов

    for decomp_name in als_decomposition_names:
        print(f"Вычисление скалярного произведения для разложения: {decomp_name} с использованием df.apply...")

        # Загружаем DataFrame эмбеддингов для текущего разложения внутри цикла
        # Это уменьшает пиковое потребление памяти при обработке нескольких decomp_names
        print(f"Загрузка эмбеддингов пользователей для {decomp_name}...")
        user_embeddings_df = pd.read_parquet(user_embeddings_path)
        print(f"user_embeddings_df: {sys.getsizeof(user_embeddings_df) // 1024 // 1024}MB")
        print(f"Загрузка эмбеддингов элементов для {decomp_name}...")
        item_embeddings_df = pd.read_parquet(item_embeddings_path)
        print(f"item_embeddings_df: {sys.getsizeof(item_embeddings_df) // 1024 // 1024}MB")

        # Надежная проверка и установка имен столбцов
        if len(user_embeddings_df.columns) == 1 and user_embeddings_df.columns[0] != decomp_name:
            user_embeddings_df.rename(columns={user_embeddings_df.columns[0]: decomp_name}, inplace=True)
            print(f"Переименован столбец пользовательских эмбеддингов в '{decomp_name}'. Текущие столбцы: {user_embeddings_df.columns.tolist()}")
        elif decomp_name not in user_embeddings_df.columns:
            print(f"Ошибка: Ожидаемый столбец '{decomp_name}' не найден в пользовательских эмбеддингах. Доступные столбцы: {user_embeddings_df.columns.tolist()}")
            raise KeyError(f"Столбец '{decomp_name}' не найден в DataFrame пользовательских эмбеддингов.")

        if len(item_embeddings_df.columns) == 1 and item_embeddings_df.columns[0] != decomp_name:
            item_embeddings_df.rename(columns={item_embeddings_df.columns[0]: decomp_name}, inplace=True)
            print(f"Переименован столбец эмбеддингов элементов в '{decomp_name}'. Текущие столбцы: {item_embeddings_df.columns.tolist()}")
        elif decomp_name not in item_embeddings_df.columns:
            print(f"Ошибка: Ожидаемый столбец '{decomp_name}' не найден в эмбеддингах элементов. Доступные столбцы: {item_embeddings_df.columns.tolist()}")
            raise KeyError(f"Столбец '{decomp_name}' не найден в DataFrame эмбеддингов элементов.")

        # Создаем отображения для быстрого поиска
        user_embedding_map = user_embeddings_df[decomp_name].to_dict()
        print(f"user_embedding_map: {sys.getsizeof(user_embedding_map) // 1024 // 1024}MB")
        del user_embeddings_df # Освобождаем память раньше
        item_embedding_map = item_embeddings_df[decomp_name].to_dict()
        print(f"item_embedding_map: {sys.getsizeof(item_embedding_map) // 1024 // 1024}MB")
        del item_embeddings_df # Освобождаем память раньше

        # Определяем размерность факторов для массива 0.0 по умолчанию, если эмбеддинг отсутствует
        factors = len(next(iter(user_embedding_map.values()))) if user_embedding_map else 0
        print(f"factors: {sys.getsizeof(factors) // 1024 // 1024}MB")
        # Применяем вспомогательную функцию построчно
        df_with_features[f'{decomp_name}_dot_product'] = df_with_features.apply(
            lambda row: _calculate_dot_product_for_row(row, user_embedding_map, item_embedding_map, factors),
            axis=1
        )
        print(f"df_with_features после скалярного произведения: {sys.getsizeof(df_with_features) // 1024 // 1024}MB")

        # Явно удаляем отображения для освобождения памяти
        del user_embedding_map
        del item_embedding_map
        print(f"Отображения эмбеддингов для {decomp_name} удалены из памяти.")

    return df_with_features

В нижних 2-х ячейках представлен фичи-инжинеринг на коллаборативных сигналах для следующего этапа. 
Разложения als_likes и als_played_ratio_pct возьмем для бейзлайна

In [32]:
als_names_2_params = {
    "als_played_ratio_pct": {"event_weights": {"likes": 0, "dislikes": 0, "unlikes": 0, "undislikes": 0}, "listen_weight": 0.01},
    "als_likes": {"event_weights": {"likes": 1, "dislikes": 0, "unlikes": -1, "undislikes": 0}, "listen_weight": 0},
    "als_dislikes": {"event_weights": {"likes": 0, "dislikes": 1, "unlikes": 0, "undislikes": -1}, "listen_weight": 0},
    "als_combined": {"event_weights": {"likes": 2, "dislikes": -2, "unlikes": -2, "undislikes": 2}, "listen_weight": 0.01}
}
als_names = als_names_2_params.keys()
als_name_2_model = dict()

user_embeddings_file = "user_embeddings.parquet"
item_embeddings_file = "item_embeddings.parquet"

for als_name in als_names_2_params:
    # Instantiate the ALSCombiner
    als_combiner = ALSCombiner(
        decomposition_name=als_name,
        factors=64,           # Number of latent factors
        regularization=0.01,  # Regularization parameter
        iterations=5,        # Number of ALS iterations
        random_state=42       # For reproducibility
    )

    # Train the ALS model on the training data
    params = als_names_2_params[als_name]
    als_combiner.fit(train_df, **params)

    # Get the generated embeddings
    user_embeddings, item_embeddings = als_combiner.get_embeddings()

    print("\n--- User Embeddings (first 5 rows) ---")
    display(user_embeddings.head())
    print("\n--- Item Embeddings (first 5 rows) ---")
    display(item_embeddings.head())


    # Save the embeddings to parquet files
    als_combiner.save_embeddings(user_embeddings_file, item_embeddings_file)

    print(f"\nUser embeddings saved to: {user_embeddings_file}")
    print(f"Item embeddings saved to: {item_embeddings_file}")

    als_name_2_model[als_name] = als_combiner

Preparing data for ALS model training with custom event weights...
Training ALS model...


100%|██████████| 5/5 [00:25<00:00,  5.10s/it]


ALS model trained successfully!

--- User Embeddings (first 5 rows) ---


,als_played_ratio_pct
uid,
100,"[-0.07659026235342026, 0.06493841111660004, 0...."
200,"[-0.024549080058932304, 0.020764781162142754, ..."
300,"[0.006262210663408041, 0.0016626655124127865, ..."
400,"[1.4711569917835732e-07, -1.9623021785264427e-..."
600,"[-0.06229391694068909, 0.04687163978815079, 0...."



--- Item Embeddings (first 5 rows) ---


...

ALS model trained successfully!

--- User Embeddings (first 5 rows) ---


,als_combined
uid,
100,"[-0.07659026235342026, 0.06493841111660004, 0...."
200,"[-0.024549080058932304, 0.020764781162142754, ..."
300,"[0.006262210663408041, 0.0016626655124127865, ..."
400,"[1.4711569917835732e-07, -1.9623021785264427e-..."
600,"[-0.06229391694068909, 0.04687163978815079, 0...."



--- Item Embeddings (first 5 rows) ---


,als_combined
item_id,
22,"[4.406048901728354e-06, 8.558946319681127e-06,..."
26,"[-0.015349420718848705, -0.006885112263262272,..."
43,"[-0.0004774877452291548, -1.023544609779492e-0..."
50,"[0.011651202104985714, -0.005607415456324816, ..."
81,"[0.00044336504652164876, -4.6559638576582074e-..."


Saving user embeddings to user_embeddings.parquet...
User embeddings saved successfully for decomposition: als_combined
Saving item embeddings to item_embeddings.parquet...
Item embeddings saved successfully for decomposition: als_combined

User embeddings saved to: user_embeddings.parquet
Item embeddings saved to: item_embeddings.parquet


In [ ]:
train_df_with_features = add_als_dot_product_features(
    df=train_df,
    als_decomposition_names=als_names,
    user_embeddings_path=user_embeddings_file,
    item_embeddings_path=item_embeddings_file
)

print("\nDataFrame with new features (first 5 rows):")
display(train_df_with_features.head())

### Бейзлайн на моделях – als_likes + top_pop и als_played_ratio_pct + top_pop

Введем класс, который будет считать целевые метрики NDCG@k, MAP@k

In [34]:
import numpy as np
import pandas as pd
import math
from collections import defaultdict
from tqdm import tqdm

class Evaluator:
    """
    Evaluates recommender models (ALSCombiner and TopPop) using NDCG@k and MAP@k metrics.
    """

    def __init__(self, test_df: pd.DataFrame, k: int = 10,
                 als_model: 'ALSCombiner' = None, top_pop_model: 'TopPop' = None):
        """
        Initializes the evaluator with test data and models.

        Args:
            test_df (pd.DataFrame): The DataFrame containing test interactions.
            k (int): The number of recommendations to consider for evaluation (e.g., k in NDCG@k, MAP@k).
            als_model (ALSCombiner, optional): An instance of ALSCombiner. Defaults to None.
            top_pop_model (TopPop, optional): An instance of TopPop. Defaults to None.
        """
        if als_model is None and top_pop_model is None:
            raise ValueError("At least one model (ALSCombiner or TopPop) must be provided for evaluation.")

        self.test_df = test_df
        self.k = k
        self.als_model = als_model
        self.top_pop_model = top_pop_model
        self.user_relevant_items = self._prepare_ground_truth()

    def _prepare_ground_truth(self) -> dict:
        """
        Prepares a dictionary mapping each user to a set of items they actually interacted with
        (considered relevant) in the test set. Filters for 'listen' events as positive interactions.
        """
        relevant_items = defaultdict(set)
        # Consider 'listen' events as relevant interactions in the test set
        test_listens = self.test_df[self.test_df['event_type'] == 'listen']
        for _, row in test_listens.iterrows():
            relevant_items[row['uid']].add(row['item_id'])
        return relevant_items

    def _calculate_ap_at_k(self, recommended_items: pd.DataFrame, relevant_items: set, k: int) -> float:
        """
        Calculates Average Precision (AP) at k for a single user.

        Args:
            recommended_items (pd.DataFrame): DataFrame of recommended items (must have 'item_id').
            relevant_items (set): Set of items the user actually interacted with (ground truth).
            k (int): Number of recommendations to consider.

        Returns:
            float: Average Precision at k.
        """
        if not relevant_items:
            return 0.0

        recommended_ids = recommended_items['item_id'].head(k).tolist()
        num_hits = 0
        sum_precisions = 0.0

        for i, item_id in enumerate(recommended_ids):
            if item_id in relevant_items:
                num_hits += 1
                sum_precisions += num_hits / (i + 1)

        return sum_precisions / len(relevant_items)

    def _calculate_ndcg_at_k(self, recommended_items: pd.DataFrame, relevant_items: set, k: int) -> float:
        """
        Calculates Normalized Discounted Cumulative Gain (NDCG) at k for a single user.

        Args:
            recommended_items (pd.DataFrame): DataFrame of recommended items (must have 'item_id').
            relevant_items (set): Set of items the user actually interacted with (ground truth).
            k (int): Number of recommendations to consider.

        Returns:
            float: Normalized Discounted Cumulative Gain at k.
        """
        if not relevant_items:
            return 0.0

        recommended_ids = recommended_items['item_id'].head(k).tolist()

        # Calculate DCG
        dcg = 0.0
        for i, item_id in enumerate(recommended_ids):
            if item_id in relevant_items:
                dcg += 1.0 / math.log2(i + 2) # i+1 is rank, so log2(rank+1)

        # Calculate IDCG
        # Create a list of relevance scores (1 for relevant, 0 for not) sorted by relevance
        ideal_relevance = [1.0] * min(len(relevant_items), k)
        idcg = 0.0
        for i, rel in enumerate(ideal_relevance):
            idcg += rel / math.log2(i + 2)

        return dcg / idcg if idcg > 0 else 0.0

    def evaluate(self) -> dict:
        """
        Evaluates the provided models and returns the average MAP@k and NDCG@k.
        If ALSCombiner cannot make predictions for a user, it falls back to TopPop recommendations.

        Returns:
            dict: A dictionary containing the evaluation results for each model.
        """
        unique_users_in_test = self.test_df['uid'].unique()
        print(f"Evaluating models for {len(unique_users_in_test)} unique users in the test set...")

        results = {}
        top_pop_recs = None # Initialize to None

        if self.top_pop_model:
            ap_scores_top_pop = []
            ndcg_scores_top_pop = []
            print(f"Generating TopPop recommendations (non-personalized)...")
            top_pop_recs = self.top_pop_model.recommend_items(N=self.k) # Generate once for all users

            for user_id in tqdm(unique_users_in_test):
                relevant_items = self.user_relevant_items.get(user_id, set())
                if not relevant_items:
                    continue

                ap_scores_top_pop.append(self._calculate_ap_at_k(top_pop_recs, relevant_items, self.k))
                ndcg_scores_top_pop.append(self._calculate_ndcg_at_k(top_pop_recs, relevant_items, self.k))

            results['TopPop'] = {
                'MAP@k': np.mean(ap_scores_top_pop) if ap_scores_top_pop else 0.0,
                'NDCG@k': np.mean(ndcg_scores_top_pop) if ndcg_scores_top_pop else 0.0
            }
            print(f"TopPop Evaluation: MAP@{self.k}={results['TopPop']['MAP@k']:.4f}, NDCG@{self.k}={results['TopPop']['NDCG@k']:.4f}")

        if self.als_model:
            ap_scores_als = []
            ndcg_scores_als = []

            perform_fallback = (self.top_pop_model is not None)
            if perform_fallback and top_pop_recs is None:
                # This means TopPop model was provided, but its evaluation block was skipped or hasn't run.
                # Generate top_pop_recs here for fallback if needed.
                print(f"Generating TopPop recommendations for ALS fallback (if not already done)...")
                top_pop_recs = self.top_pop_model.recommend_items(N=self.k)

            if perform_fallback:
                print(f"Generating ALS recommendations with TopPop fallback for each user...")
            else:
                print(f"Generating ALS recommendations (no TopPop fallback available)...")

            for user_id in tqdm(unique_users_in_test):
                relevant_items = self.user_relevant_items.get(user_id, set())
                if not relevant_items:
                    continue

                als_recs = pd.DataFrame() # Initialize als_recs as empty DataFrame
                try:
                    als_recs = self.als_model.recommend_items(user_id, N=self.k)
                except Exception as e:
                    print(f"Warning: ALS model failed to recommend for user {user_id}. Error: {e}. Falling back to TopPop if available.")
                    # als_recs remains an empty DataFrame, which will trigger the fallback logic below.

                # Apply fallback logic
                if als_recs.empty and perform_fallback: # If ALS couldn't recommend and fallback is enabled
                    current_recs = top_pop_recs
                elif not als_recs.empty: # ALS made recommendations
                    current_recs = als_recs
                else: # als_recs is empty and no fallback was possible/enabled
                      # This user won't contribute to ALS scores if no recommendations can be made
                    continue # Skip this user for ALS evaluation if no recommendations

                ap_scores_als.append(self._calculate_ap_at_k(current_recs, relevant_items, self.k))
                ndcg_scores_als.append(self._calculate_ndcg_at_k(current_recs, relevant_items, self.k))

            # Store the result, naming it appropriately based on whether fallback was used
            if perform_fallback:
                results['ALSCombiner_with_TopPop_Fallback'] = {
                    'MAP@k': np.mean(ap_scores_als) if ap_scores_als else 0.0,
                    'NDCG@k': np.mean(ndcg_scores_als) if ndcg_scores_als else 0.0
                }
                print(f"ALSCombiner_with_TopPop_Fallback Evaluation: MAP@{self.k}={results['ALSCombiner_with_TopPop_Fallback']['MAP@k']:.4f}, NDCG@{self.k}={results['ALSCombiner_with_TopPop_Fallback']['NDCG@k']:.4f}")
            else:
                results['ALSCombiner'] = {
                    'MAP@k': np.mean(ap_scores_als) if ap_scores_als else 0.0,
                    'NDCG@k': np.mean(ndcg_scores_als) if ndcg_scores_als else 0.0
                }
                print(f"ALSCombiner Evaluation: MAP@{self.k}={results['ALSCombiner']['MAP@k']:.4f}, NDCG@{self.k}={results['ALSCombiner']['NDCG@k']:.4f}")

        return results


Обучим TopPop

In [35]:
# Instantiate the TopPop recommender
top_pop_recommender = TopPop()

# Fit the model on the training data
top_pop_recommender.fit(train_df)

# Get top 10 popular items (for any user, as it's a non-personalized model)
top_items = top_pop_recommender.recommend_items(N=10)

print("\n--- Top 10 Popular Items ---")
display(top_items)

Fitting TopPop model...
TopPop model fitted successfully!

--- Top 10 Popular Items ---


,item_id,popularity_score
0,5463340,34928
1,5635052,34329
2,5862961,33040
3,6901374,32878
4,9378983,31520
5,8213481,31075
6,906358,30921
7,3542184,27358
8,2859641,26520
9,3033749,25861


Оценим модели

In [ ]:
# Define the k for evaluation (e.g., top 10 recommendations)
K_VALUE = 10

als_combiners = ['als_played_ratio_pct', 'als_likes']
results = dict()

for als_name in als_combiners:
    # Instantiate the Evaluator with both models and the test_df
    # Ensure als_combiner and top_pop_recommender are already initialized from previous cells
    als_combiner = als_name_2_model[als_name]
    evaluator = Evaluator(
        test_df=test_df,
        k=K_VALUE,
        als_model=als_combiner,          # Assuming als_combiner is available from previous steps
        top_pop_model=top_pop_recommender # Assuming top_pop_recommender is available from previous steps
    )

    # Run the evaluation
    evaluation_results = evaluator.evaluate()
    results[als_name] = evaluation_results

    print("\n--- Evaluation Results ---")
    for model_name, metrics in evaluation_results.items():
        print(f"Model: {model_name}")
        for metric_name, score in metrics.items():
            print(f"  {metric_name}: {score:.4f}")

In [37]:
print(results)

{'als_played_ratio_pct': {'TopPop': {'MAP@k': np.float64(0.004505924669268508), 'NDCG@k': np.float64(0.21026747828713926)}, 'ALSCombiner_with_TopPop_Fallback': {'MAP@k': np.float64(0.0033266313573488237), 'NDCG@k': np.float64(0.16195073800158186)}}, 'als_likes': {'TopPop': {'MAP@k': np.float64(0.004505924669268508), 'NDCG@k': np.float64(0.21026747828713926)}, 'ALSCombiner_with_TopPop_Fallback': {'MAP@k': np.float64(0.0005639242243955638), 'NDCG@k': np.float64(0.021149602084952968)}}}


Получили такие результаты:
1. als_played_ratio_pct: NDCG@10 = 0.16, MAP@10 = 0.0033
2. als_likes: NDCG@10 = 0.021, MAP@10 = 0.0056
3. TopPop: NDCG@10 = 0.21, MAP@10 = 0.045

Результат контринтуитивен, в следующем этапе планируется разобраться, почему TopPop по метрикам обходит ALS-разложения, а также на als и контентных фичах обучить более тяжелую модель ранжирования